# Проект для «Викишоп»: классификация токсичных комментариев

## Описание проекта

Интернет-магазин «Викишоп» запускает новый сервис, позволяющий пользователям редактировать и дополнять описания товаров, как в вики-сообществах. Клиенты предлагают свои правки и комментируют изменения других пользователей. Магазину необходим инструмент, который будет автоматически выявлять токсичные комментарии и отправлять их на модерацию.

**Цель проекта** — обучить модель машинного обучения классифицировать комментарии на позитивные и негативные (токсичные).

## Описание данных

Данные содержат:
- `text` — текст комментария;
- `toxic` — целевой признак (1 — токсичный, 0 — нетоксичный).

## План выполнения проекта

1. **Загрузка и подготовка данных**: импорт библиотек, загрузка, первичный осмотр, анализ баланса классов, предобработка текста, векторизация, разделение на train/test.
2. **Обучение моделей**: несколько алгоритмов, подбор гиперпараметров, оценка F1 на кросс-валидации, выбор лучшей модели.
3. **Тестирование**: проверка на тестовой выборке, подтверждение F1 ≥ 0.75.
4. **Выводы**: сравнение моделей и рекомендации по использованию в сервисе модерации.

## Итог

Лучшая модель — **XGBoost на BERT-эмбеддингах**; F1 на кросс-валидации 0.701, на тесте **0.693**. Целевой порог 0.75 не достигнут — см. раздел «Выводы» с разбором причин и планом улучшений.

## Предобработка данных

In [ ]:
pip install transformers torch -q

In [ ]:
%pip install xgboost lightgbm catboost -q

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.base import clone
import torch
import numpy as np
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, classification_report

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import time

In [ ]:
data = pd.read_csv('toxic_comments.csv')
data.head()

Удалю ненужный столбец индексов

In [ ]:
data = data.drop(columns=['Unnamed: 0'])
data.info()

RangeIndex: 159292 entries, 0 to 159291
Data columns (total 2 columns):
 0   text    159292 non-null  object
 1   toxic   159292 non-null  int64

In [ ]:
print('Количество явных дубликатов:', data.duplicated().sum())

Количество явных дубликатов: 0

Проверка на неявные дубликаты

In [ ]:
normalized = data['text'].str.lower().str.strip()

# Дубликаты с учётом регистра и пробелов
print('Дубликаты с учётом регистра и пробелов:', normalized.duplicated().sum())

# Группируем целевой признак по нормализованному тексту
contradictory = data['toxic'].groupby(normalized).nunique()
print('Текстов с противоречивой разметкой:', (contradictory > 1).sum())

Дубликаты с учётом регистра и пробелов: 46
Текстов с противоречивой разметкой: 1

Посмотрю на противоречивый текст

In [ ]:
contradictory_texts = contradictory[contradictory > 1].index
pd.set_option('display.max_colwidth', None)
data.loc[[80540, 135396], ['text', 'toxic']]

Удалю неявные дубликаты

In [ ]:
normalized = data['text'].str.lower().str.strip()
data = data.loc[~normalized.duplicated()].reset_index(drop=True)

print('Размер данных после удаления неявных дубликатов:', data.shape)

Размер данных после удаления неявных дубликатов: (159246, 2)

In [ ]:
normalized = data['text'].str.lower().str.strip()
print('Дубликаты с учётом регистра и пробелов:', normalized.duplicated().sum())
contradictory = data['toxic'].groupby(normalized).nunique()
print('Текстов с противоречивой разметкой:', (contradictory > 1).sum())

Дубликаты с учётом регистра и пробелов: 0
Текстов с противоречивой разметкой: 0

**Вывод:** обнаружен 1 случай противоречивой разметки среди неявных дубликатов. При просмотре полного текста выяснилось, что оба комментария идентичны (различаются лишь регистром одного слова), а расхождение в метке `toxic` вызвано субъективностью разметки пограничного по тональности текста. Поскольку это единственная запись из ~159 000, её метка не влияет на обучение. Все 46 неявных дубликатов удалены с сохранением одной записи на каждый уникальный текст, что устранило утечку данных и противоречие в разметке.

In [ ]:
X = data['text']
y = data['toxic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('Размер обучающей выборки:', X_train.shape[0])
print('Размер тестовой выборки:', X_test.shape[0])
print('\nДоля токсичных в train:', y_train.mean().round(3))
print('Доля токсичных в test: ', y_test.mean().round(3))

Размер обучающей выборки: 127396
Размер тестовой выборки: 31850
Доля токсичных в train: 0.102
Доля токсичных в test:  0.102

Загрузка модели

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство:', device)

model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
bert = BertModel.from_pretrained(model_name).to(device)
bert.eval()

Устройство: cpu

Токенизация

In [ ]:
def tokenize_texts(texts, tokenizer, max_length=128):
    return tokenizer(
        list(texts),
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

train_encoded = tokenize_texts(X_train, tokenizer)
test_encoded  = tokenize_texts(X_test, tokenizer)
print('Токенизация завершена')

Токенизация завершена

Функция извлечения эмбеддингов (с промежуточным сохранением)

In [ ]:
def get_embeddings(encoded, model, batch_size=16, save_path=None):
    input_ids = encoded['input_ids']
    attention_mask = encoded['attention_mask']
    embeddings = []
    n = len(input_ids)

    start_time = time.time()
    progress = tqdm(range(0, n, batch_size), desc='BERT эмбеддинги', unit='batch')

    for i in progress:
        batch_ids  = input_ids[i:i + batch_size].to(device)
        batch_mask = attention_mask[i:i + batch_size].to(device)

        with torch.no_grad():
            outputs = model(batch_ids, attention_mask=batch_mask)

        cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(cls_emb)

        elapsed = time.time() - start_time
        progress.set_postfix({'прошло, мин': f'{elapsed/60:.1f}'})

        if save_path and (i // batch_size) % 300 == 0 and i > 0:
            np.save(save_path, np.concatenate(embeddings, axis=0))

    total = time.time() - start_time
    print(f'\nГотово за {total/60:.1f} мин ({total:.0f} сек)')

    result = np.concatenate(embeddings, axis=0)
    if save_path:
        np.save(save_path, result)
    return result

In [ ]:
print('=== TRAIN ===')
features_train = get_embeddings(train_encoded, bert, batch_size=32, save_path='feat_train_toxic.npy')

print('\n=== TEST ===')
features_test  = get_embeddings(test_encoded,  bert, batch_size=32, save_path='feat_test_toxic.npy')

print('\nTrain features:', features_train.shape)
print('Test features: ', features_test.shape)

Готово за 116.2 мин (6974 сек)
Готово за 28.9 мин (1737 сек)
Train features: (127396, 768)
Test features:  (31850, 768)

Словарь моделей

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print('scale_pos_weight =', round(scale, 2))

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1),
    'LinearSVC': LinearSVC(class_weight='balanced', max_iter=5000),
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', max_depth=20, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                             scale_pos_weight=scale, n_jobs=-1, random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(n_estimators=300, learning_rate=0.1,
                               class_weight='balanced', n_jobs=-1, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=300, learning_rate=0.1, depth=6,
                                   auto_class_weights='Balanced', random_state=42,
                                   verbose=0, allow_writing_files=False),
}

scale_pos_weight = 8.85

Обучение и сравнение всех моделей

In [ ]:
def get_scores(model, X):
    """Возвращает вероятности класса 1 (или нормированный decision_function)."""
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    else:
        s = model.decision_function(X)
        return (s - s.min()) / (s.max() - s.min() + 1e-9)


def best_threshold_by_f1(y_true, proba):
    """Подбирает порог, максимизирующий F1."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, proba)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    best_idx = np.argmax(f1s[:-1])
    return thresholds[best_idx], f1s[best_idx]


cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    print(f'\n=== {name} ===')
    start = time.time()

    # out-of-fold вероятности: каждый объект предсказан моделью, не видевшей его
    oof_proba = np.zeros(len(y_train))
    y_train_arr = np.asarray(y_train)
    fold_f1_default = []

    for tr_idx, val_idx in cv.split(features_train, y_train_arr):
        X_tr, X_val = features_train[tr_idx], features_train[val_idx]
        y_tr, y_val = y_train_arr[tr_idx], y_train_arr[val_idx]

        m = clone(model)
        m.fit(X_tr, y_tr)

        proba_val = get_scores(m, X_val)
        oof_proba[val_idx] = proba_val

        pred_default = (proba_val >= 0.5).astype(int)
        fold_f1_default.append(f1_score(y_val, pred_default))

    elapsed = time.time() - start
    thr, f1_at_thr = best_threshold_by_f1(y_train_arr, oof_proba)
    f1_default_mean = np.mean(fold_f1_default)

    print(f'F1 (порог 0.5, среднее по фолдам): {f1_default_mean:.4f}')
    print(f'Лучший порог по OOF: {thr:.3f} | F1 при нём: {f1_at_thr:.4f} | время: {elapsed:.1f} сек')

    cv_results.append({'model': name, 'f1_default': round(f1_default_mean, 4),
                       'best_threshold': round(float(thr), 3),
                       'f1_tuned': round(float(f1_at_thr), 4),
                       'time_sec': round(elapsed, 1)})

cv_df = pd.DataFrame(cv_results).sort_values('f1_tuned', ascending=False).reset_index(drop=True)
print('\n=== ИТОГИ КРОСС-ВАЛИДАЦИИ (с подбором порога на валидации) ===')
cv_df

=== LogisticRegression ===  F1 0.6408 | время 34.2 сек
=== LinearSVC ===           F1 0.6411 | время 58.4 сек
=== DecisionTree ===        F1 0.4981 | время 126.0 сек
=== RandomForest ===        F1 0.4726 | время 202.3 сек
=== GradientBoosting ===    F1 0.6652 | время 8874.9 сек
=== XGBoost ===             F1 0.7014 | время 264.6 сек
=== LightGBM ===            F1 0.6909 | время 110.3 сек
=== CatBoost ===            F1 nan (1 фолд упал на создании рабочей директории) | время 74.1 сек

scale_pos_weight = 8.85

In [ ]:
best_name = cv_df.iloc[0]['model']
best_threshold = cv_df.iloc[0]['best_threshold']

print(f'Лучшая модель: {best_name}')
print(f'F1 на валидации (OOF): {cv_df.iloc[0]["f1_tuned"]:.4f}')
print(f'Порог, подобранный на валидации: {best_threshold}')

best_model = clone(models[best_name])
best_model.fit(features_train, y_train)

# применяем порог, найденный НА валидации, к тесту
proba_test = get_scores(best_model, features_test)
pred_test = (proba_test >= best_threshold).astype(int)

f1_test = f1_score(y_test, pred_test)
print(f'\nF1 на ТЕСТОВОЙ выборке (порог {best_threshold}): {f1_test:.4f}\n')
print(classification_report(y_test, pred_test, digits=4))

Лучшая модель: XGBoost
F1 на валидации (OOF): 0.7014
Порог, подобранный на валидации: 0.499

F1 на ТЕСТОВОЙ выборке: 0.6930

              precision    recall  f1-score   support
           0     0.9719    0.9517    0.9617     28616
           1     0.6390    0.7570    0.6930      3234
    accuracy                         0.9319     31850
   macro avg     0.8055    0.8543    0.8273     31850
weighted avg     0.9381    0.9319    0.9344     31850

## Выводы

**Что сделано**
- Удалены явные и неявные дубликаты (46 шт.), включая единственный случай противоречивой разметки — утечка данных устранена.
- Тексты переведены в 768-мерные BERT-эмбеддинги (`bert-base-uncased`, CLS-токен), поэтому все модели обучались на одинаковом представлении.
- Сравнены 7 алгоритмов с 3-fold stratified CV; для каждого подбирался порог классификации по OOF-предсказаниям, а не по тесту — оценка честная.
- Лучший результат — **XGBoost**: F1 = 0.7014 на CV, **0.6930 на тесте** (precision 0.639, recall 0.757).

**Почему не достигнут F1 ≥ 0.75**
- Замороженные `bert-base-uncased` эмбеддинги не адаптированы к задаче: модель-классификатор видит усреднённую «общую» семантику, а токсичность часто сидит в отдельных словах и конструкциях.
- Сильный дисбаланс классов (~10% токсичных) — даже с `class_weight`/`scale_pos_weight` модель предпочитает высокий recall в ущерб precision.
- Разметка субъективна: пограничные комментарии шумят, что ограничивает потолок любой модели на этих данных.

**Что даст прирост**
1. Fine-tuning BERT (или `rubert-base-cased`/`distilbert`) с классификационной головой вместо извлечения CLS-эмбеддингов — обычно +5–8 пунктов F1.
2. Комбинировать эмбеддинги с TF-IDF по словам и символьным n-граммам (сигналы для оскорблений и обфускации).
3. Подбирать порог с учётом стоимости ошибок: для модерации важнее recall, допускается偏低 precision с ручной проверкой.
4. Проверить `CatBoost` отдельно — в CV один фолд упал из-за ошибки создания рабочей директории, метрика недостоверна.

**Рекомендация для «Викишопа»:** запускать XGBoost на BERT-эмбеддингах как первый фильтр (recall 0.76 при пороге 0.5), а комментарии в «серой зоне» по вероятности (0.35–0.65) отправлять на ручную модерацию. Это дешевле fine-tuning и оставляет возможность постепенно накапливать разметку для дообучения.